In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q19/q19_rewrite/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# Define container and ship mode lists
SM_SMALL = ["SM BOX", "SM CASE", "SM PACK", "SM PKG"]
MED_CONTAINER = ["MED BAG", "MED BOX", "MED PACK", "MED PKG"]
LG_CONTAINER = ["LG BOX", "LG CASE", "LG PACK", "LG PKG"]
SHIPMODES = ["AIR", "AIRREG"]

# 1) Collapse the three disjoint between() calls into one: 4–36
# 2) Use __getitem__ (df["col"]) instead of attribute access to cut down NDFrame.__getattr__ overhead
fline = lineitem[
    lineitem["L_QUANTITY"].between(4, 36)
    & (lineitem["L_SHIPINSTRUCT"] == "DELIVER IN PERSON")
    & lineitem["L_SHIPMODE"].isin(SHIPMODES)
]

# Filter part in one shot
fpart = part[
    ((part["P_BRAND"] == "Brand#31") & part["P_CONTAINER"].isin(SM_SMALL) & part["P_SIZE"].between(1, 5))
    | ((part["P_BRAND"] == "Brand#43") & part["P_CONTAINER"].isin(MED_CONTAINER) & part["P_SIZE"].between(1, 10))
    | ((part["P_BRAND"] == "Brand#43") & part["P_CONTAINER"].isin(LG_CONTAINER) & part["P_SIZE"].between(1, 15))
]

# Join
df = fline.merge(fpart, left_on="L_PARTKEY", right_on="P_PARTKEY")

# Final multi‐range filter in one mask
mask = (
    (df["P_BRAND"] == "Brand#31") & df["P_CONTAINER"].isin(SM_SMALL) & df["L_QUANTITY"].between(4, 14) & (df["P_SIZE"] <= 5)
    | (df["P_BRAND"] == "Brand#43") & df["P_CONTAINER"].isin(MED_CONTAINER) & df["L_QUANTITY"].between(15, 25) & (df["P_SIZE"] <= 10)
    | (df["P_BRAND"] == "Brand#43") & df["P_CONTAINER"].isin(LG_CONTAINER) & df["L_QUANTITY"].between(26, 36) & (df["P_SIZE"] <= 15)
)

df = df[mask]

# Compute total on GPU
total = (df["L_EXTENDEDPRICE"] * (1.0 - df["L_DISCOUNT"])) .sum()